In [ ]:
from pathlib import Path
import runpy

relative_paths = (
    "dataset", "services/ComputerVision/dataset",
    "app/services/ComputerVision/dataset",
    "back-end/ia/app/services/ComputerVision/dataset",
)
base_path = next(
    (parent / relative for parent in (Path.cwd(), *Path.cwd().parents)
     for relative in relative_paths
     if (parent / relative / "data.yaml").is_file()
     and (parent / relative).parent.joinpath("split_dataset.py").is_file()),
    None,
)
if base_path is None:
    raise RuntimeError("Execute este notebook dentro do repositório WaterPath_TCC.")
split = runpy.run_path(str(base_path.parent / "split_dataset.py"))["split_dataset"]
print(split(base_path))


In [ ]:
!pip install ultralytics

In [ ]:
import ultralytics

print(ultralytics.__version__)

In [ ]:
from ultralytics import YOLO

model = YOLO(str(base_path.parent / "yolov8n.pt"))
model.train(
    data=str(base_path / "data.yaml"),
    epochs=20,
    imgsz=800,
    project=str(base_path.parent / "runs/detect"),
)
training_run = Path(model.trainer.save_dir)
best_weights = Path(model.trainer.best)


In [ ]:
# Avalia os melhores pesos no conjunto de teste reservado.
model = YOLO(str(best_weights))
model.val(data=str(base_path / "data.yaml"), split="test")


In [ ]:
from IPython.display import Image, display

display(Image(filename=str(training_run / "results.png")))
display(Image(filename=str(training_run / "BoxP_curve.png")))


In [ ]:
# Exemplo de inferência em uma imagem do conjunto de teste.
test_image = (base_path / "test.txt").read_text(encoding="utf-8").splitlines()[0]
image_path = base_path / test_image.removeprefix("./")
result = model.predict(str(image_path), show=True, save=False, conf=0.25)
